# Gradient-conflict analysis across models/seeds

In [ ]:
import gc
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader

from breast_datasets.report_generation_dataset import ReportGenerationDataset
from components.processing.misc import div_255, repeat_rgb_channels
from data_preprocessing.medical_mappings import birads_mapping
from models.amber.amber import Amber

MODEL_RUNS = {
    "amber_md": [
        "out/1s3pwzqy/ckpt.pt",
        "out/3h3s05id/ckpt.pt",
        "out/2idpvmra/ckpt.pt",
    ],
}

SETS = {
    "US": {
        "csv_path": "../data/report_generation_split/us-rg-train.csv",
        "imgs_path": "../data/report_generation_split/us-rg-train.bin",
    },
    "MR": {
        "csv_path": "../data/report_generation_split/mr-rg-train.csv",
        "imgs_path": "../data/report_generation_split/mr-rg-train.bin",
    },
    "MG": {
        "csv_path": "../data/report_generation_split/mg-rg-train.csv",
        "imgs_path": "../data/report_generation_split/mg-rg-train.bin",
    },
    "CESM": {
        "csv_path": "../data/report_generation_split/cesm-rg-train.csv",
        "imgs_path": "../data/report_generation_split/cesm-rg-train.bin",
    },
}

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]
IMG_RESIZE = 560
IMGS_SHAPE = (512, 512)
BATCH_SIZE = 4
MAX_SAMPLES = 3000  # per modality; None = full split
NUM_WORKERS = 4  # set to 0 for easier debugging

SHARED_NAME_FILTERS = ("encoder",)  # , "img_projector", "decoder")

REPORT_COL_CANDIDATES = (
    "report",
    "findings",
    "impression",
    "text",
    "report_text",
    "processed_report",
    "target",
)

OUTPUT_DIR = Path("gradient_conflict_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
def normalize_model_runs(model_runs):
    if isinstance(model_runs, dict):
        return {str(k): list(v) for k, v in model_runs.items()}
    return {"model": list(model_runs)}


def build_loader(csv_path, imgs_path):
    tfm = T.Compose(
        [
            T.ToImage(),
            T.Lambda(repeat_rgb_channels),
            T.Resize(IMG_RESIZE, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(IMG_RESIZE),
            T.Lambda(div_255),
        ]
    )

    ds = ReportGenerationDataset(
        imgs_path=imgs_path,
        csv_path=csv_path,
        imgs_shape=IMGS_SHAPE,
        transform=tfm,
        return_birads=True,
        return_modalities=True,
        return_exam_type=False,
        return_origin_dataset=True,
        return_exam_ids=True,
    )

    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda"),
        shuffle=False,
    )


def birads_to_str(b):
    b = int(b)
    return birads_mapping[b] if b in birads_mapping else str(b)


def cosine(a, b):
    denom = a.norm() * b.norm()
    if float(denom) == 0.0:
        return np.nan
    return float(torch.dot(a, b) / (denom + 1e-12))


def mean_std_str(values, decimals=3):
    values = pd.Series(values, dtype="float64").dropna()
    if len(values) == 0:
        return "nan ± nan"
    mean = values.mean()
    std = values.std(ddof=1) if len(values) > 1 else 0.0
    return f"{mean:+.{decimals}f} ± {std:.{decimals}f}"

In [ ]:
def load_model_and_shared_params(chkpt_path):
    model = Amber.from_pretrained(chkpt_path=chkpt_path, device=device, eval_mode=True)
    model.eval()
    model.tokenizer.eval()

    shared_named_params = [
        (n, p)
        for n, p in model.named_parameters()
        if p.requires_grad and any(f in n for f in SHARED_NAME_FILTERS)
    ]

    if not shared_named_params:
        raise RuntimeError(
            f"No trainable parameters matched SHARED_NAME_FILTERS={SHARED_NAME_FILTERS}"
        )

    print(f"loaded {chkpt_path}")
    print(f"shared param tensors: {len(shared_named_params)}")
    print(f"shared grad dim: {sum(p.numel() for _, p in shared_named_params):,}")

    return model, [p for _, p in shared_named_params]


@torch.enable_grad()
def modality_grad(model, shared_params, loader):
    normalize = T.Normalize(mean=NORM_MEAN, std=NORM_STD)
    pad_id = model.tokenizer.get_pad_token_id()

    model.zero_grad(set_to_none=True)
    n_samples = 0
    n_tokens = 0

    for imgs, report, birads, modalities, *_ in loader:
        imgs = imgs.to(device, non_blocking=True)
        if imgs.dim() == 5:
            imgs = imgs.squeeze(1)

        reports = [str(r).lower() for r in report]
        birads_str = [birads_to_str(b) for b in birads]

        inp, mask, tgt = model.tokenizer.encode_batch(
            reports, birads_str, list(modalities)
        )
        inp = inp.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)

        logits = model(
            normalize(imgs),
            inp,
            mask,
            flat_logits=True,
            process_image=True,
        )

        flat_tgt = tgt.view(-1)
        loss = F.cross_entropy(
            logits,
            flat_tgt,
            ignore_index=pad_id,
            reduction="sum",
        )

        # Scale by token count only to keep gradients numerically stable.
        # Cosine is scale-invariant, so this does not change the direction.
        valid_tokens = int((flat_tgt != pad_id).sum().item())
        loss = loss / max(valid_tokens, 1)
        loss.backward()

        n_samples += imgs.size(0)
        n_tokens += valid_tokens

        if MAX_SAMPLES is not None and n_samples >= MAX_SAMPLES:
            break

    grad_chunks = [
        p.grad.detach().flatten().float().cpu()
        for p in shared_params
        if p.grad is not None
    ]

    if not grad_chunks:
        raise RuntimeError(
            "No gradients were produced for the selected shared parameters."
        )

    g = torch.cat(grad_chunks)
    print(f"  {n_samples} samples | {n_tokens} target tokens | grad dim {g.numel():,}")
    return g, n_samples, n_tokens

In [ ]:
def run_one_checkpoint(model_name, seed_idx, chkpt_path):
    model, shared_params = load_model_and_shared_params(chkpt_path)

    grads = {}
    counts = {}

    for set_name, s in SETS.items():
        print(f"gradient for {model_name} seed={seed_idx} set={set_name}")
        loader = build_loader(s["csv_path"], s["imgs_path"])
        grads[set_name], n_samples, n_tokens = modality_grad(
            model, shared_params, loader
        )
        counts[set_name] = {"n_samples": n_samples, "n_tokens": n_tokens}

    rows = []
    for a, b in combinations(grads.keys(), 2):
        rows.append(
            {
                "model": model_name,
                "seed_idx": seed_idx,
                "checkpoint": chkpt_path,
                "pair": f"{a}-{b}",
                "a": a,
                "b": b,
                "cosine": cosine(grads[a], grads[b]),
                "n_samples_a": counts[a]["n_samples"],
                "n_samples_b": counts[b]["n_samples"],
                "n_tokens_a": counts[a]["n_tokens"],
                "n_tokens_b": counts[b]["n_tokens"],
            }
        )

    del grads, shared_params, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return rows


def run_all_checkpoints():
    all_rows = []
    model_runs = normalize_model_runs(MODEL_RUNS)

    for model_name, checkpoints in model_runs.items():
        for seed_idx, chkpt_path in enumerate(checkpoints, start=1):
            print("=" * 100)
            print(f"model={model_name} | seed_idx={seed_idx} | checkpoint={chkpt_path}")
            all_rows.extend(run_one_checkpoint(model_name, seed_idx, chkpt_path))

    per_seed_df = pd.DataFrame(all_rows)
    per_seed_path = OUTPUT_DIR / "grad_cosine_per_seed.csv"
    per_seed_df.to_csv(per_seed_path, index=False)
    print(f"saved {per_seed_path}")

    return per_seed_df


per_seed_df = run_all_checkpoints()
per_seed_df

In [ ]:
def summarize_results(per_seed_df):
    summary_rows = []

    for (model_name, pair), g in per_seed_df.groupby(["model", "pair"], sort=False):
        vals = g["cosine"].dropna()
        summary_rows.append(
            {
                "model": model_name,
                "pair": pair,
                "n": int(vals.shape[0]),
                "mean": float(vals.mean()) if len(vals) else np.nan,
                "std": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
                "mean_pm_std": mean_std_str(vals),
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    summary_path = OUTPUT_DIR / "grad_cosine_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"saved {summary_path}")

    return summary_df


def make_summary_matrix(summary_df, model_name):
    names = list(SETS.keys())
    mat = pd.DataFrame("", index=names, columns=names)

    for x in names:
        mat.loc[x, x] = "+1.000 ± 0.000"

    lookup = {
        tuple(row["pair"].split("-")): row["mean_pm_std"]
        for _, row in summary_df[summary_df["model"] == model_name].iterrows()
    }

    for a, b in combinations(names, 2):
        val = lookup.get((a, b), lookup.get((b, a), ""))
        mat.loc[a, b] = val
        mat.loc[b, a] = val

    matrix_path = OUTPUT_DIR / f"grad_cosine_summary_matrix_{model_name}.csv"
    mat.to_csv(matrix_path)
    print(f"saved {matrix_path}")
    return mat


summary_df = summarize_results(per_seed_df)
display(summary_df)

for model_name in summary_df["model"].unique():
    print("\n" + "=" * 100)
    print(model_name)
    display(make_summary_matrix(summary_df, model_name))